In [1]:
import pandas as pd
import numpy as np

print("Libraries loaded successfully")


Libraries loaded successfully


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("Libraries loaded successfully")

Libraries loaded successfully


In [3]:
rfm = pd.read_csv("../data/customer_rfm_analysis.csv")

print("Shape:", rfm.shape)

print("\nColumns:")
print(rfm.columns.tolist())

rfm.head()

Shape: (5942, 9)

Columns:
['Customer ID', 'Recency', 'Frequency', 'Monetary', 'R Score', 'F Score', 'M Score', 'RFM Score', 'Customer Segment']


,Customer ID,Recency,Frequency,Monetary,R Score,F Score,M Score,RFM Score,Customer Segment
0,12346.0,326,17,-64.68,2,5,1,251,At Risk
1,12347.0,3,8,5633.32,5,4,5,545,VIP Customer
2,12348.0,76,5,2019.40,3,3,4,334,Regular Customer
3,12349.0,19,5,4404.54,5,3,5,535,Regular Customer
4,12350.0,311,1,334.40,2,1,2,212,Regular Customer


In [4]:
print("Missing values:")
print(rfm.isnull().sum())

print("\nCustomer segments:")
print(rfm["Customer Segment"].value_counts())

print("\nData types:")
print(rfm.dtypes)

Missing values:
Customer ID         0
Recency             0
Frequency           0
Monetary            0
R Score             0
F Score             0
M Score             0
RFM Score           0
Customer Segment    0
dtype: int64

Customer segments:
Customer Segment
Regular Customer    2637
VIP Customer        1317
At Risk              837
Loyal Customer       689
Recent Customer      462
Name: count, dtype: int64

Data types:
Customer ID         float64
Recency               int64
Frequency             int64
Monetary            float64
R Score               int64
F Score               int64
M Score               int64
RFM Score             int64
Customer Segment        str
dtype: object


In [5]:
rfm["Risk"] = (
    rfm["Customer Segment"]
    .apply(lambda x: 1 if x == "At Risk" else 0)
)

print("Risk distribution:")
print(rfm["Risk"].value_counts())

print("\nRisk percentage:")
print(rfm["Risk"].value_counts(normalize=True) * 100)

Risk distribution:
Risk
0    5105
1     837
Name: count, dtype: int64

Risk percentage:
Risk
0    85.913834
1    14.086166
Name: proportion, dtype: float64


In [6]:
features = [
    "Recency",
    "Frequency",
    "Monetary",
    "R Score",
    "F Score",
    "M Score"
]

X = rfm[features]
y = rfm["Risk"]

print("Features:")
print(X.columns.tolist())

print("\nX shape:", X.shape)
print("y shape:", y.shape)

Features:
['Recency', 'Frequency', 'Monetary', 'R Score', 'F Score', 'M Score']

X shape: (5942, 6)
y shape: (5942,)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (4753, 6)
Testing data: (1189, 6)


In [8]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

print("Random Forest model trained successfully.")

Random Forest model trained successfully.


In [9]:
y_pred = model.predict(X_test)

print("Predictions generated successfully.")

print("\nFirst 20 predictions:")
print(y_pred[:20])

Predictions generated successfully.

First 20 predictions:
[0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0]


In [10]:
accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Model Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1022
           1       1.00      1.00      1.00       167

    accuracy                           1.00      1189
   macro avg       1.00      1.00      1.00      1189
weighted avg       1.00      1.00      1.00      1189



In [11]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[1022    0]
 [   0  167]]


In [12]:
feature_importance = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance)

     Feature  Importance
0    Recency    0.307205
3    R Score    0.272785
4    F Score    0.227002
1  Frequency    0.156188
2   Monetary    0.026300
5    M Score    0.010520


In [13]:
rfm["Predicted_Risk"] = model.predict(X)

rfm["Risk_Probability"] = model.predict_proba(X)[:, 1]

rfm.head()

,Customer ID,Recency,Frequency,Monetary,R Score,F Score,M Score,RFM Score,Customer Segment,Risk,Predicted_Risk,Risk_Probability
0,12346.0,326,17,-64.68,2,5,1,251,At Risk,1,1,1.0
1,12347.0,3,8,5633.32,5,4,5,545,VIP Customer,0,0,0.0
2,12348.0,76,5,2019.40,3,3,4,334,Regular Customer,0,0,0.0
3,12349.0,19,5,4404.54,5,3,5,535,Regular Customer,0,0,0.0
4,12350.0,311,1,334.40,2,1,2,212,Regular Customer,0,0,0.0


In [14]:
rfm["Risk_Level"] = pd.cut(
    rfm["Risk_Probability"],
    bins=[0, 0.30, 0.70, 1.00],
    labels=["Low Risk", "Medium Risk", "High Risk"],
    include_lowest=True
)

print(rfm["Risk_Level"].value_counts())

Risk_Level
Low Risk       5105
High Risk       837
Medium Risk       0
Name: count, dtype: int64


In [15]:
high_risk_customers = rfm.sort_values(
    by="Risk_Probability",
    ascending=False
)

high_risk_customers[
    [
        "Customer ID",
        "Recency",
        "Frequency",
        "Monetary",
        "Customer Segment",
        "Risk_Probability",
        "Risk_Level"
    ]
].head(20)

,Customer ID,Recency,Frequency,Monetary,Customer Segment,Risk_Probability,Risk_Level
0,12346.0,326,17,-64.68,At Risk,1.0,High Risk
461,12807.0,383,3,269.95,At Risk,1.0,High Risk
2937,15283.0,391,4,135.43,At Risk,1.0,High Risk
1884,14230.0,436,3,979.80,At Risk,1.0,High Risk
1194,13540.0,468,3,632.39,At Risk,1.0,High Risk
465,12811.0,261,5,1033.90,At Risk,1.0,High Risk
1195,13541.0,424,3,717.90,At Risk,1.0,High Risk
467,12813.0,502,3,1687.80,At Risk,1.0,High Risk
1196,13542.0,377,18,3970.77,At Risk,1.0,High Risk
2932,15278.0,562,4,496.16,At Risk,1.0,High Risk


In [16]:
output_file = "../output/customer_risk_prediction.csv"

rfm.to_csv(output_file, index=False)

print("Saved successfully:")
print(output_file)

Saved successfully:
../output/customer_risk_prediction.csv


In [17]:
risk_summary = (
    rfm.groupby("Risk_Level", observed=True)
    .agg(
        Customers=("Customer ID", "count"),
        Average_Recency=("Recency", "mean"),
        Average_Frequency=("Frequency", "mean"),
        Average_Monetary=("Monetary", "mean")
    )
    .reset_index()
)

risk_summary

,Risk_Level,Customers,Average_Recency,Average_Frequency,Average_Monetary
0,Low Risk,5105,175.900294,7.807248,2995.340127
1,High Risk,837,371.530466,5.997611,1621.363252


In [18]:
risk_summary.to_csv(
    "../output/customer_risk_summary.csv",
    index=False
)

print("Risk summary saved successfully.")

Risk summary saved successfully.
